In [2]:
import os

from dotenv import load_dotenv

from azure.identity import DefaultAzureCredential, get_bearer_token_provider

from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import AzureSearch

load_dotenv()


credential = DefaultAzureCredential()

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default",
)

False

In [4]:
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=os.environ["AZURE_AI_ENDPOINT"],
    azure_ad_token_provider=token_provider,
    api_version="2024-10-21",
    azure_deployment=os.environ["AZURE_AI_EMBEDDING_DEPLOYMENT"],
    chunk_size=1,
)

In [9]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("wine-ratings.csv")
documents = loader.load()

In [ ]:
acs = AzureSearch(
    azure_search_endpoint=os.getenv("SEARCH_SERVICE_NAME"),
    azure_search_key=os.getenv("SEARCH_API_KEY"),
    index_name=os.getenv("SEARCH_INDEX_NAME"),
    embedding_function=embeddings.embed_query,
)

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

# for testing, we will only add 1000 rows to the index
documents = documents[:1000]
docs = text_splitter.split_documents(documents)

acs.add_documents(documents=docs)

In [5]:
docs = acs.similarity_search_with_relevance_scores(
    query="What is the best Cabernet Sauvignon wine in Napa Valley above 94 points",
    k=5,
)
print(docs[0][0].page_content)
print(dir(docs[0][0]))

: 20
name: 1849 Declaration Napa Valley Cabernet Sauvignon 2014
grape: 
region: Napa Valley, California
variety: Red Wine
rating: 91.0
notes: The palate is robust with flavors of dark blueberry, blackberry, traces of red currant, and subtle sweet oak from the barrel. This wine is fruit forward, full-bodied and spreads richly across the palate with soft velvety tannins and a long-lasting finish.
['Config', '__abstractmethods__', '__annotations__', '__class__', '__class_vars__', '__config__', '__custom_root_type__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__exclude_fields__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_validators__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__include_fields__', '__init__', '__init_subclass__', '__iter__', '__json_encoder__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__post_root_validators__', '__pre_root_validators__', '__pretty__', '__private_attributes__', '__reduce__', '__reduce_ex

In [8]:
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["AZURE_AI_ENDPOINT"] + "/openai/v1",
    api_key=token_provider,
)


context = "\n\n".join(
    [doc[0].page_content for doc in docs]
)


messages = [
    {
        "role": "system",
        "content": (
            "You are a wine assistant. "
            "Answer using only the provided context."
        ),
    },
    {
        "role": "user",
        "content": (
            f"""
Context:
{context}

Question:
What is the best wine in Oregon above 92 points?
"""
        ),
    },
]


response = client.chat.completions.create(
    model=os.environ["AZURE_CHAT_DEPLOYMENT"],
    messages=messages,
)


print(response.choices[0].message.content)


{'choices': [{'finish_reason': 'stop',
              'index': 0,
              'message': {'content': 'I apologize for the confusion, but I '
                                     "don't have access to real-time wine "
                                     'ratings and reviews. It would be best to '
                                     'refer to professional wine rating '
                                     'websites or consult with a sommelier for '
                                     'specific recommendations on Oregon Pinot '
                                     'Noir wines above 94 points. They will '
                                     'have the most up-to-date and accurate '
                                     'information for you.',
                          'role': 'assistant'}}],
 'created': 1703696035,
 'id': 'chatcmpl-8aRSFAMFjRLXsPOGHajprdnUPTeuu',
 'model': 'gpt-35-turbo',
 'object': 'chat.completion',
 'usage': {'completion_tokens': 66,
           'prompt_tokens': 154,
